# wandb-init-run — worked example 1: Open a wandb run with project, name, and config

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `wandb-init-run`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The three essential keyword arguments for `wandb.init()` are `project` (which project on the wandb dashboard to log to), `name` (the human-readable label for this specific run), and `config` (a dict or dataclass capturing all hyperparameters). Passing `config=args` lets wandb snapshot every field of your args dataclass alongside the logged metrics.

## Worked solution

**Step 1 — gather the three kwargs.**
We need `project`, `name`, and `config`. In ARENA's pattern, `project` and `name` come from fields on the `args` dataclass. The `config` argument receives the entire `args` object — wandb will inspect its fields automatically.

**Step 2 — call wandb.init.**
We call `wandb.init(project=args.wandb_project, name=args.wandb_name, config=args)`. The keyword argument names must match exactly — positional arguments don't work reliably across wandb versions.

**Step 3 — return the run handle.**
`wandb.init()` returns a run handle (a `Run` object in real wandb, a `MagicMock` return value in tests). Returning it lets the caller access the run object if needed (e.g., to log artifacts or update the run summary).

In [ ]:
import sys
from unittest.mock import MagicMock
from dataclasses import dataclass
sys.modules.setdefault('wandb', MagicMock())
import wandb

@dataclass
class Args:
    lr: float = 1e-3
    batch_size: int = 64
    n_epochs: int = 10
    wandb_project: str = 'linear-classifier'
    wandb_name: str = 'baseline-run'

def open_wandb_run(args):
    """Open a wandb run with the canonical three kwargs."""
    return wandb.init(
        project=args.wandb_project,
        name=args.wandb_name,
        config=args,
    )

# Exercise it
wandb.init.reset_mock()
args = Args(lr=5e-4, batch_size=128)
run = open_wandb_run(args)
print('init called:', wandb.init.called)
call_kwargs = wandb.init.call_args.kwargs
print('project:', call_kwargs['project'])
print('name:', call_kwargs['name'])
print('config is args:', call_kwargs['config'] is args)